# Topic Modelling with LDA

In [1]:
import pandas as pd
import re
import spacy
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

In [2]:
nlp = spacy.load("de_core_news_sm")

In [3]:
df = pd.read_csv("../data/wikipedia_text.csv", sep = ",")

In [4]:
df.head()

,name,genderLabel,article,title,text
0,Andre Geim,männlich,https://de.wikipedia.org/wiki/Andre_Geim,Andre Geim,Sir Andre Konstantin Geim\n (russisch Андре́й ...
1,Bruce Beutler,männlich,https://de.wikipedia.org/wiki/Bruce_Beutler,Bruce Beutler,Bruce Alan Beutler (* 29. Dezember 1957 in Chi...
2,Benjamin List,männlich,https://de.wikipedia.org/wiki/Benjamin_List,Benjamin List,Benjamin „Ben“ List (* 11. Januar 1968 in Fran...
3,Roland Kirstein,männlich,https://de.wikipedia.org/wiki/Roland_Kirstein,Roland Kirstein,Roland Kirstein (* 10. August 1965 in Bremen; ...
4,Patrick Cramer,männlich,https://de.wikipedia.org/wiki/Patrick_Cramer,Patrick Cramer,Patrick Cramer (* 3. Februar 1969 in Stuttgart...


## Configuration

In [5]:
CSV_PATH   = "../data/wikipedia_text.csv"
NUM_TOPICS = 5
NUM_WORDS  = 10
RANDOM_STATE = 42

## Preprocessing

In [6]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z\säöüß]", " ", text)
    doc  = nlp(text)
    tokens = [t.lemma_ for t in doc if t.text not in stop_words and len(t.text) > 2 and not t.is_stop]
    return " ".join(tokens)

In [14]:
stop_words = set(stopwords.words("german")) | set(stopwords.words("english"))

# domain-specific noise words
custom_stopwords = {"isbn", "hrsg", "einzelnachweise", "abgerufen", "seite", "band", "online"}
stop_words = stop_words | custom_stopwords

In [8]:
df["clean_text"] = df["text"].astype(str).apply(preprocess)

## LDA per Gender

In [15]:
def run_lda(texts, label):
    print(f"\n{'='*60}")
    print(f"  GENDER: {label.upper()}  ({len(texts)} documents)")
    print(f"{'='*60}")

    vectorizer = CountVectorizer(max_df=0.95, min_df=1,
                                 max_features=1000, stop_words=list(stop_words))
    dtm = vectorizer.fit_transform(texts)
    vocab = vectorizer.get_feature_names_out()

    lda = LatentDirichletAllocation(
        n_components=NUM_TOPICS,
        random_state=RANDOM_STATE,
        max_iter=20,
        learning_method="batch",
    )
    lda.fit(dtm)

    # Print top words per topic
    for i, component in enumerate(lda.components_):
        top_words = [vocab[idx] for idx in component.argsort()[:-NUM_WORDS-1:-1]]
        print(f"  Topic {i+1}: {', '.join(top_words)}")

    # Document-topic distribution
    doc_topics = lda.transform(dtm)
    dominant   = doc_topics.argmax(axis=1) + 1      # 1-indexed
    topic_dist = pd.Series(dominant).value_counts().sort_index()
    print(f"\n  Document counts per dominant topic:")
    for t, count in topic_dist.items():
        print(f"    Topic {t}: {count} document(s)")

    return lda, vectorizer

In [16]:
models = {}
for gender in sorted(df["genderLabel"].unique()):
    texts          = df[df["genderLabel"] == gender]["clean_text"].tolist()
    models[gender] = run_lda(texts, label=str(gender))



  GENDER: MÄNNLICH  (383 documents)
  Topic 1: universität, deutsch, institut, international, professor, mitglied, münchen, deutschland, weblink, leben
  Topic 2: universität, deutsch, berlin, geschichte, wien, digital, literatur, institut, wissenschaftlich, verlag
  Topic 3: jantke, berlin, universität, klaus, system, digital, international, springer, spring, learning
  Topic 4: universität, deutsch, institut, sorbisch, zimmermann, bern, niedersorbisch, language, gotthelf, dresden
  Topic 5: university, professor, russisch, institut, research, universität, erhalten, preis, mitglied, american

  Document counts per dominant topic:
    Topic 1: 111 document(s)
    Topic 2: 137 document(s)
    Topic 3: 10 document(s)
    Topic 4: 11 document(s)
    Topic 5: 114 document(s)

  GENDER: WEIBLICH  (227 documents)
  Topic 1: berlin, sozial, arbeit, pädagogik, bildung, hartmann, pädagogisch, gender, universität, jutta
  Topic 2: russisch, universität, institut, wissenschaft, dissertation, leh

## Visualization